# Kham River Restoration – Parametric Analysis
**Objective:** Identify which Indian rivers can replicate the Kham River restoration model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Data – Kham River Baseline

In [ ]:
kham_pre = pd.DataFrame({
    'Parameter': ['DO', 'BOD', 'COD', 'TDS', 'TSS', 'Fecal Coliform'],
    'Value':     [2.1,  48.0,  162.0, 1450.0, 380.0, 9200],
    'Unit':      ['mg/L','mg/L','mg/L','mg/L','mg/L','MPN/100mL'],
    'CPCB_Limit':[5.0,  3.0,   150.0, 500.0,  100.0, 500]
})

kham_post = pd.DataFrame({
    'Parameter': ['DO', 'BOD', 'COD', 'TDS', 'TSS'],
    'Pre':  [2.1,  48.0, 162.0, 1450.0, 380.0],
    'Post': [4.8,  12.0,  85.0,  680.0, 120.0]
})

kham_outcomes = pd.DataFrame({
    'Outcome': ['Riparian Restored (acres)', 'Households Connected', 'Saplings Planted',
                'Garbage Points Eliminated', 'Community Participants', 'Flood-Free Years'],
    'Value':   [54, 25000, 100000, 110, 1000000, 2]
})

print('Kham Pre-Restoration Water Quality:')
kham_pre

## 2. Data – Candidate Rivers

In [ ]:
rivers = pd.DataFrame({
    'river':   ['Kham (Reference)', 'Nag River', 'Mula-Mutha', 'Cooum River',
                'Sukhna Choe', 'Shivna River', 'Sabarmati Tribs', 'Rispana-Bindal'],
    'state':   ['Maharashtra', 'Maharashtra', 'Maharashtra', 'Tamil Nadu',
                'Punjab/Haryana', 'Maharashtra', 'Gujarat', 'Uttarakhand'],
    'DO':      [2.1,  1.8,  2.4,  1.2,  3.1,  3.2,  2.0,  2.8],
    'BOD':     [48.0, 52.0, 42.0, 68.0, 35.0, 28.0, 55.0, 38.0],
    'COD':     [162.0,178.0,155.0,210.0,120.0, 95.0,195.0,130.0],
    'TDS':     [1450.,1620.,980., 2100.,870., 720., 2400.,950.],
    'TSS':     [380., 410., 290., 520., 260., 180., 450., 340.],
    'fecal_coliform': [9200,11000,8500,24000,6200,4200,15000,7800],
    'type':    ['seasonal_intermittent','seasonal_intermittent','seasonal_with_dam_regulation',
                'seasonal_urban','seasonal_rivulet','seasonal_intermittent',
                'seasonal_intermittent','seasonal_himalayan'],
    'governance_readiness': ['high','high','high','medium','medium_high','high','medium','medium'],
    'has_initiatives': [True, True, True, True, True, True, True, True],
    'matching_params': [
        ['urban_river','seasonal_flow','legacy_waste','community_identity_potential'],
        ['seasonal_flow','urban_river','community_identity_potential','restoration_discourse_ongoing'],
        ['seasonal_flow','urban_river','encroachment','community_identity_potential'],
        ['seasonal_flow','urban_river','legacy_waste','encroachment'],
        ['seasonal_flow','urban_river','siltation_issues','strong_civic_identity'],
        ['seasonal_flow'],
        ['seasonal_flow','semi_arid_catchment','encroachment'],
        ['seasonal_flow','urban_river','encroachment','community_memory','heavy_metal_contamination']
    ]
})
rivers

## 3. Parameter Scoring

In [ ]:
# Water Quality Index
wqi_weights = {'DO': 0.20, 'BOD': 0.25, 'COD': 0.15, 'TDS': 0.15, 'TSS': 0.10, 'fecal_coliform': 0.15}
cpcb = {'DO': 5.0, 'BOD': 3.0, 'COD': 150.0, 'TDS': 500.0, 'TSS': 100.0, 'fecal_coliform': 500.0}
kham_ref = {'BOD': 48.0, 'COD': 162.0, 'fecal_coliform': 9200.0}

def compute_wqi(row):
    wqi = 0.0
    for p, w in wqi_weights.items():
        v = row[p]
        if p == 'DO':
            q = min(100.0, 100.0 * v / cpcb[p])
        else:
            q = min(100.0, 100.0 * cpcb[p] / v) if v > 0 else 100.0
        wqi += w * q
    return round(wqi, 2)

def compute_degradation(mp_list):
    indicators = {'encroachment': 0.20, 'siltation_issues': 0.15, 'urban_river': 0.30,
                  'legacy_waste': 0.20, 'heavy_metal_contamination': 0.15}
    score = sum(w for k, w in indicators.items() if any(k in mp for mp in mp_list))
    return round(min(score, 1.0), 2)

def compute_waste(row):
    score = (0.5 * min(1.0, row['BOD'] / kham_ref['BOD']) +
             0.3 * min(1.0, row['COD'] / kham_ref['COD']) +
             0.2 * min(1.0, row['fecal_coliform'] / kham_ref['fecal_coliform']))
    return round(score, 2)

def compute_governance(row):
    gmap = {'high': 0.80, 'medium_high': 0.65, 'medium': 0.50, 'low': 0.20}
    score = gmap.get(row['governance_readiness'], 0.2)
    if row['has_initiatives']:
        score = min(1.0, score + 0.15)
    return round(score, 2)

def compute_community(mp_list):
    indicators = {'community_identity_potential': 0.30, 'community_memory': 0.25,
                  'strong_civic_identity': 0.25, 'restoration_discourse_ongoing': 0.20}
    score = sum(w for k, w in indicators.items() if any(k in mp for mp in mp_list))
    return round(min(score, 1.0), 2)

def compute_seasonality(row):
    tmap = {'seasonal_intermittent': 1.0, 'seasonal_himalayan': 0.85, 'seasonal_rivulet': 0.80,
            'seasonal_urban': 0.75, 'seasonal_with_dam_regulation': 0.60, 'perennial': 0.20}
    score = tmap.get(row['type'], 0.3)
    if 'seasonal_flow' in row['matching_params']:
        score = min(1.0, score + 0.05)
    return round(score, 2)

params = pd.DataFrame()
params['river']             = rivers['river']
params['state']             = rivers['state']
params['WQI']               = rivers.apply(compute_wqi, axis=1)
params['Physical_Eco']      = rivers['matching_params'].apply(compute_degradation)
params['Waste_Load']        = rivers.apply(compute_waste, axis=1)
params['Governance']        = rivers.apply(compute_governance, axis=1)
params['Community']         = rivers['matching_params'].apply(compute_community)
params['Seasonality']       = rivers.apply(compute_seasonality, axis=1)
params = params.set_index('river')
params

## 4. Normalise and Compute Similarity

In [ ]:
param_cols = ['WQI', 'Physical_Eco', 'Waste_Load', 'Governance', 'Community', 'Seasonality']
weights    = np.array([0.25, 0.15, 0.15, 0.20, 0.10, 0.15])

norm = params[param_cols].copy()
norm['WQI'] = 100.0 - norm['WQI']   # invert: higher pollution = higher score

for col in param_cols:
    lo, hi = norm[col].min(), norm[col].max()
    norm[col] = (norm[col] - lo) / (hi - lo) if hi > lo else 1.0

vals = norm.values
n = len(vals)

ed = np.zeros((n, n))
cs = np.zeros((n, n))
md = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        diff = vals[i] - vals[j]
        ed[i, j] = np.sqrt(np.sum(weights * diff**2))
        ni, nj = np.linalg.norm(vals[i]), np.linalg.norm(vals[j])
        cs[i, j] = np.dot(vals[i], vals[j]) / (ni * nj) if ni > 0 and nj > 0 else 0.0
        md[i, j] = np.sum(weights * np.abs(diff))

rivers_list = norm.index.tolist()
kham_idx = 0

ed_kham = ed[kham_idx]
cs_kham = cs[kham_idx]
md_kham = md[kham_idx]

ed_norm = ed_kham / ed_kham.max() if ed_kham.max() > 0 else ed_kham
md_norm = md_kham / md_kham.max() if md_kham.max() > 0 else md_kham

ALPHA, BETA, GAMMA = 0.40, 0.35, 0.25
ri = ALPHA * cs_kham + BETA * (1 - ed_norm) + GAMMA * (1 - md_norm)

def classify(r):
    if r >= 0.85: return 'Excellent Match'
    elif r >= 0.70: return 'Strong Match'
    elif r >= 0.55: return 'Moderate Match'
    elif r >= 0.40: return 'Partial Match'
    else: return 'Low Match'

ranking = pd.DataFrame({
    'river': rivers_list,
    'state': params['state'].values,
    'RI': ri,
    'Cosine_Sim': cs_kham,
    'Euclidean_Dist': ed_kham,
    'Manhattan_Dist': md_kham
})
ranking = ranking[ranking['river'] != 'Kham (Reference)'].copy()
ranking = ranking.sort_values('RI', ascending=False).reset_index(drop=True)
ranking.index += 1
ranking['Category'] = ranking['RI'].apply(classify)
ranking

## 5. Export Results to CSV

In [ ]:
import os
os.makedirs('output', exist_ok=True)
params.to_csv('output/parameter_matrix.csv')
norm.to_csv('output/normalised_matrix.csv')
ranking.to_csv('output/replicability_ranking.csv', index=False)
print('Saved to output/')

## 6. Charts

In [ ]:
# --- Chart 1: Pre vs Post Restoration Line Chart ---
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(kham_post))
ax.plot(x, kham_post['Pre'],  marker='o', color='#D55E00', label='Pre-Restoration',  linewidth=2)
ax.plot(x, kham_post['Post'], marker='s', color='#009E73', label='Post-Restoration', linewidth=2)
ax.set_xticks(x)
ax.set_xticklabels(kham_post['Parameter'])
ax.set_ylabel('Concentration (mg/L)')
ax.set_title('Kham River: Water Quality Before and After Restoration')
ax.legend()
plt.tight_layout()
plt.savefig('output/line_pre_post.png', dpi=150)
plt.show()

In [ ]:
# --- Chart 2: BOD Histogram across all rivers ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(rivers['BOD'], bins=7, color='#56B4E9', edgecolor='white', linewidth=1.2)
ax.axvline(3.0, color='red', linestyle='--', label='CPCB Limit (3 mg/L)')
ax.set_xlabel('BOD (mg/L)')
ax.set_ylabel('Number of Rivers')
ax.set_title('Distribution of BOD Values Across All Candidate Rivers')
ax.legend()
plt.tight_layout()
plt.savefig('output/hist_bod.png', dpi=150)
plt.show()

In [ ]:
# --- Chart 3: Replicability Index Bar (horizontal) ---
colors_map = {'Excellent Match': '#009E73', 'Strong Match': '#56B4E9',
              'Moderate Match': '#E69F00', 'Partial Match': '#D55E00', 'Low Match': '#CC79A7'}
sorted_r = ranking.sort_values('RI')
bar_colors = [colors_map.get(c, '#999') for c in sorted_r['Category']]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(sorted_r['river'], sorted_r['RI'], color=bar_colors, edgecolor='white', linewidth=1)
for i, (_, row) in enumerate(sorted_r.iterrows()):
    ax.text(row['RI'] + 0.01, i, f"{row['RI']:.3f}", va='center', fontsize=9)
ax.axvline(0.70, color='gray', linestyle=':', linewidth=1)
ax.axvline(0.55, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('Replicability Index (RI)')
ax.set_title('Replicability Index Ranking – Candidate Rivers vs Kham Model')
ax.set_xlim(0, 1.1)
plt.tight_layout()
plt.savefig('output/ri_ranking.png', dpi=150)
plt.show()

In [ ]:
# --- Chart 4: Parameter Weights Pie Chart ---
param_labels = ['Water Quality (25%)', 'Physical/Eco (15%)', 'Waste Load (15%)',
                'Governance (20%)', 'Community (10%)', 'Seasonality (15%)']
colors_pie = ['#E69F00','#56B4E9','#009E73','#0072B2','#D55E00','#CC79A7']

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    weights, labels=param_labels, colors=colors_pie,
    autopct='%1.0f%%', startangle=90,
    wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontweight('bold')
ax.set_title('Parameter Weights for Replicability Index', fontsize=13)
plt.tight_layout()
plt.savefig('output/weights_pie.png', dpi=150)
plt.show()

In [ ]:
# --- Chart 5: Cosine Similarity Line Chart (vs Kham) ---
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ranking['river'], ranking['Cosine_Sim'], marker='o', color='#0072B2', linewidth=2)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Cosine Similarity')
ax.set_title('Cosine Similarity to Kham River (Parametric Profile Match)')
ax.set_ylim(0.5, 1.05)
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig('output/cosine_line.png', dpi=150)
plt.show()

In [ ]:
# --- Chart 6: TDS Histogram ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(rivers['TDS'], bins=6, color='#009E73', edgecolor='white', linewidth=1.2)
ax.axvline(500.0, color='red', linestyle='--', label='CPCB Limit (500 mg/L)')
ax.set_xlabel('TDS (mg/L)')
ax.set_ylabel('Number of Rivers')
ax.set_title('Distribution of TDS Values Across All Candidate Rivers')
ax.legend()
plt.tight_layout()
plt.savefig('output/hist_tds.png', dpi=150)
plt.show()

In [ ]:
# --- Chart 7: Category Distribution Pie ---
cat_counts = ranking['Category'].value_counts()
cat_colors = [colors_map.get(c, '#999') for c in cat_counts.index]

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(cat_counts.values, labels=cat_counts.index, colors=cat_colors,
       autopct='%1.0f%%', startangle=90,
       wedgeprops=dict(edgecolor='white', linewidth=2))
ax.set_title('Distribution of Replicability Categories')
plt.tight_layout()
plt.savefig('output/category_pie.png', dpi=150)
plt.show()

In [ ]:
# --- Chart 8: Metric Comparison Line Chart (RI, Cosine, 1-ED) ---
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(ranking))
ed_plot = 1 - (ranking['Euclidean_Dist'] / ranking['Euclidean_Dist'].max())
ax.plot(x, ranking['RI'],         marker='o', label='Replicability Index', linewidth=2, color='#0072B2')
ax.plot(x, ranking['Cosine_Sim'], marker='s', label='Cosine Similarity',   linewidth=2, color='#E69F00')
ax.plot(x, ed_plot,               marker='^', label='1 - ED (Proximity)',  linewidth=2, color='#009E73')
ax.set_xticks(x)
ax.set_xticklabels(ranking['river'], rotation=25, ha='right')
ax.set_ylabel('Score')
ax.set_title('Multi-Metric Similarity Comparison Across Candidate Rivers')
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig('output/metrics_line.png', dpi=150)
plt.show()

## 7. Summary

The **Nag River (Maharashtra)** ranks #1 with the highest Replicability Index, making it the strongest candidate for direct application of Kham restoration methodologies.